# 3. Train GRU12 và GRU20
Attach output notebook 02. Chọn GPU và Internet On. Không cần video gốc hay MediaPipe model ở bước train.

In [ ]:
import sys, os, json, subprocess, urllib.request, zipfile
from pathlib import Path
INPUT, WORK = Path('/kaggle/input'), Path('/kaggle/working')
REV = 'a64400b'
archive = WORK/'code20.zip'
urllib.request.urlretrieve(f'https://github.com/toafn11/driver-monitoring-system/archive/{REV}.zip', archive)
destination = WORK/f'code_{REV}'
with zipfile.ZipFile(archive) as z:
    for name in z.namelist():
        if not (destination/name).resolve().is_relative_to(destination.resolve()):
            raise ValueError('Unsafe archive path')
    z.extractall(destination)
CODE = next(destination.rglob('prepare20.py')).parents[1]
os.chdir(CODE)
sys.path.insert(0, str(CODE))
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(CODE/'requirements-kaggle20.txt')],check=True)
# BEGIN VERIFIED PROJECT ACTIVATION
import importlib, hashlib
EXPECTED_CODE_HASHES = {'training/preflight20.py': '20c0fde7e542fd6cd9aa7623d6254dd57b8b45d770c0d53a3384055e2eb8c8e6', 'training/prepare20.py': '6861700d4d574c12cc5c9a88dd4cace5a6d9faba6f8f1e35d3fdb0521a1bf78a', 'training/video_timestamps20.py': '84d5f81cc1f2bd16312ba573af251fe4ec1899e37a26882e7c80bc97168de32e'}
for relative, expected in EXPECTED_CODE_HASHES.items():
    actual = hashlib.sha256((CODE/relative).read_bytes()).hexdigest()
    if actual != expected:
        raise RuntimeError(f'Code integrity mismatch: {relative}; re-import the notebook')
# Replacing sys.path alone does not replace modules retained by a notebook kernel.
for name in list(sys.modules):
    if name in ('training','core','models') or name.startswith(('training.','core.','models.')):
        del sys.modules[name]
sys.path.insert(0, str(CODE))
importlib.invalidate_caches()
for name in ('training.preflight20','training.prepare20','training.video_timestamps20'):
    module = importlib.import_module(name)
    actual_path = Path(module.__file__).resolve()
    expected_path = (CODE/Path(*name.split('.'))).with_suffix('.py').resolve()
    if actual_path != expected_path:
        raise RuntimeError(f'Wrong module loaded: {name}: {actual_path}')
    print('Verified module:', name, actual_path)
print('Verified timestamp version:', sys.modules['training.video_timestamps20'].TIMESTAMP_VERSION)
# END VERIFIED PROJECT ACTIVATION
print('Code revision VERIFIED:', REV)


In [ ]:
import torch
from training.train20 import train, load_data
candidates = [p.parent for p in INPUT.rglob('metadata.json') if p.parent.name=='processed20_v4']
if len(candidates)!=1: raise ValueError('Attach exactly one completed processed20_v4 dataset')
DATA = candidates[0]
splits, metadata = load_data(DATA)
print('Device:', 'cuda' if torch.cuda.is_available() else 'cpu')
print('Preprocessing:', metadata['preprocessing'])
for name, split in splits.items(): print(name, split['X'].shape)
del splits
# A new directory each execution avoids mistaking interrupted runs for completed runs.
from datetime import datetime, timezone
RUN = WORK/('experiment_'+datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f'))
RUN.mkdir()
for dim in (12,20):
    train(DATA,RUN/f'gru{dim}_seed42',feature_dim=dim,cell='gru',hidden=64,epochs=40,seed=42)
(RUN/'complete.json').write_text(json.dumps(dict(revision=REV, dataset=str(DATA))))


In [ ]:
best_runs = []
for dim in (12,20):
    run = RUN/f'gru{dim}_seed42'
    history = json.loads((run/'history.json').read_text())
    best = max(history,key=lambda e:e['validation']['macro_f1'])
    best_runs.append((best['validation']['macro_f1'],run))
    print(run.name,best['validation'])
CHOSEN = max(best_runs,key=lambda item:item[0])[1]/'best.pt'
print('Selected by validation macro-F1:', CHOSEN)


## Test cuối cùng
Chỉ bật sau khi đã chốt thí nghiệm bằng validation. Không dùng kết quả test để chỉnh model. Nếu chạy phiên mới, đặt DATA và CHOSEN tới output đã attach; không chạy lại cell train chỉ để lấy biến.

In [ ]:
RUN_FINAL_TEST = False
if RUN_FINAL_TEST:
    from training.train20 import evaluate
    evaluate(DATA, CHOSEN, RUN/'test_final')
else:
    print('Test held back; checkpoints and validation reports are saved.')
